# Evaluación de EfficientNetB0 — 5 clases de residuos

Este notebook mide qué tan bien clasifica el modelo **EfficientNetB0** ya entrenado, usando el set de **prueba** (`data/raw/waste_classification/test`). No entrena nada: solo carga el modelo y analiza sus predicciones.

**Qué se obtiene:**
1.  Accuracy y loss globales.
2.  Matriz de confusión y reporte de clasificación (precision, recall, F1).
3.  Ejemplos visuales de imágenes mal clasificadas.
4.  Accuracy por clase.

**Modelo evaluado:** `models/EfficientNetB0/modelo_residuos_EfficientNetB0_v1.keras` (generado por `entrenamiento_EfficientNetB0.ipynb`).

> ⚠️ Este notebook está guardado **sin salidas** (nunca se ejecutó completo en esta copia). `tests/comparar_modelos.py` lee las métricas de las salidas de los notebooks de evaluación, así que hay que ejecutarlo antes de correr ese script.

> 📁 Este notebook vive en `notebooks/EfficientNetB0/`, por eso las rutas parten de `Path("../..")` (dos niveles arriba = raíz del proyecto) y los modelos se leen/guardan en `models/EfficientNetB0/`.

### Importación de librerías

Herramientas que se usan para evaluar el modelo:

* **`numpy`:** operaciones con arreglos (por ejemplo, elegir la clase con mayor probabilidad con `argmax`).
* **`matplotlib` y `seaborn`:** gráficas; `seaborn.heatmap` dibuja la matriz de confusión.
* **`tensorflow` / `load_model`:** carga el modelo `.keras` ya entrenado.
* **`ImageDataGenerator`:** lee las imágenes de prueba desde carpetas.
* **`classification_report` y `confusion_matrix` (scikit-learn):** calculan las métricas detalladas por clase.
* **`pathlib.Path`:** rutas que funcionan igual en Windows, Linux y Mac.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

### Configuración de rutas

Definimos de dónde se leen los datos y el modelo, y dónde se guardan las gráficas.

* **`BASE_DIR = Path("../..")`:** raíz del proyecto (dos niveles arriba de `notebooks/EfficientNetB0/`).
* **`DATA_DIR`:** `data/raw/waste_classification/test`, el set de **prueba**. Son imágenes que el modelo nunca usó para ajustar sus pesos.
* **`MODEL_PATH`:** `models/EfficientNetB0/modelo_residuos_EfficientNetB0_v1.keras`, generado por `entrenamiento_EfficientNetB0.ipynb`.
* **`FIGURES_DIR`:** `reports/figures/`, donde se guardan las figuras para el reporte.
* **`FIGURES_DIR.mkdir(...)`:** crea la carpeta de figuras si no existe.

In [ ]:
# --- 1. CONFIGURACIÓN DE RUTAS (Compatible Windows/Linux) ---
BASE_DIR = Path("../..")
DATA_DIR = BASE_DIR / "data" / "raw" / "waste_classification" / "test"
MODELS_DIR = BASE_DIR / "models" / "EfficientNetB0"
MODEL_PATH = MODELS_DIR / "modelo_residuos_EfficientNetB0_v1.keras"
FIGURES_DIR = BASE_DIR / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

### Verificación de archivos

Antes de hacer cualquier cálculo comprobamos que todo lo necesario existe, para obtener un error claro en vez de uno confuso más adelante.

1.  Si no existe la carpeta de prueba, se detiene con `FileNotFoundError` indicando la ruta buscada.
2.  Si no existe el modelo, se detiene pidiendo **ejecutar primero el notebook de entrenamiento**.

In [ ]:
# Verificamos que existan
if not DATA_DIR.exists():
    raise FileNotFoundError(f"No se encontraron los datos en: {DATA_DIR}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo en: {MODEL_PATH}. ¡Ejecuta el entrenamiento primero!")

print(f"Rutas verificadas.")
print(f"Cargando modelo desde: {MODEL_PATH}")

### Carga del modelo entrenado

`load_model()` lee el archivo `.keras` y reconstruye el modelo completo (EfficientNetB0 + cabecera de clasificación) con los pesos aprendidos. A partir de aquí el modelo queda listo para hacer predicciones; **no se vuelve a entrenar**.

In [ ]:
# --- 2. CARGA DEL MODELO ---
model = load_model(MODEL_PATH)
print("Modelo EfficientNetB0 cargado exitosamente.")

### Preparación del set de prueba

Creamos un generador que lee las imágenes de `test/` exactamente igual que durante el entrenamiento, pero **sin Data Augmentation**: queremos evaluar las fotos tal como son.

* **`IMG_SIZE = (224, 224)`:** mismo tamaño que en el entrenamiento.
* **`BATCH_SIZE = 16`:** cuántas imágenes se procesan a la vez. En evaluación no afecta el resultado, solo la velocidad y la memoria usada.
* **Preprocesamiento:** no se aplica `rescale` porque EfficientNetB0 trae su reescalado y normalización dentro del modelo (igual que en el entrenamiento).
* **`shuffle=False` (¡importante!):** mantiene las imágenes en orden. Así la posición *i* de las predicciones corresponde a la etiqueta real *i* de `test_gen.classes`; si se mezclaran, las métricas saldrían mal.
* **Nombres de las clases:** se obtienen de `test_gen.class_indices` (orden alfabético de las carpetas).

In [ ]:
# --- 3. PREPARACIÓN DE DATOS (Solo Validación) ---
# Usamos las mismas dimensiones que en el entrenamiento
IMG_SIZE = (224, 224)
BATCH_SIZE = 16 # Podemos usar un batch más grande para inferencia

# Solo necesitamos re-escalar, no hacemos Data Augmentation para evaluar.
# NO se aplica rescale=1./255 porque EfficientNetB0 tiene preprocesamiento interno.
datagen = ImageDataGenerator()

print("Generando set de validación...")
test_gen = datagen.flow_from_directory(
    str(DATA_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # ¡IMPORTANTE! No mezclar para que coincidan las etiquetas con la predicción
)

class_labels = list(test_gen.class_indices.keys())
print(f"Clases detectadas: {len(class_labels)}")

### Evaluación global (`model.evaluate`)

Pasamos todo el set de prueba por el modelo y obtenemos dos números resumen:

* **Pérdida (loss):** el error promedio (`categorical_crossentropy`). Cuanto más bajo, mejor.
* **Precisión (accuracy):** porcentaje de imágenes clasificadas correctamente.

Es la "calificación final" del modelo, pero no dice **en qué** se equivoca. Para eso están las celdas siguientes.

In [ ]:
print("Evaluando el modelo con los datos de prueba...")
loss, accuracy = model.evaluate(test_gen)

print(f"\nResultados Finales:")
print(f"   Pérdida (Loss): {loss:.4f}")
print(f"   Precisión (Accuracy): {accuracy*100:.2f}%")

### Predicciones, matriz de confusión y reporte de clasificación

**1. Predicciones:** `model.predict` devuelve, para cada imagen, una probabilidad por clase. Con `np.argmax` nos quedamos con la clase más probable (`y_pred`) y la comparamos con la etiqueta real (`y_true`).

**2. Matriz de confusión:**
* Cada **fila** es la clase real y cada **columna** la clase que predijo el modelo.
* La **diagonal** son los aciertos; todo lo que está fuera son errores.
* *Ejemplo de lectura:* si en la fila `vidrio` hay un 31 en la columna `plastico`, significa que 31 vidrios se confundieron con plástico.
* Se guarda en `reports/figures/matriz_confusion_EfficientNetB0.png`.

**3. Reporte de clasificación (por clase):**
* **Precision:** de todo lo que el modelo dijo que era de esa clase, ¿cuánto lo era realmente?
* **Recall:** de todo lo que realmente era de esa clase, ¿cuánto encontró el modelo?
* **F1-score:** promedio armónico de precision y recall (un solo número que equilibra ambas).
* **Support:** cuántas imágenes reales hay de esa clase.
* **Macro avg** promedia las clases por igual; **weighted avg** pondera por la cantidad de imágenes de cada clase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("Generando predicciones detalladas...")
predictions = model.predict(test_gen)
y_pred = np.argmax(predictions, axis=1) # Tomamos la clase con mayor probabilidad
y_true = test_gen.classes               # Las etiquetas reales

# Obtenemos los nombres de las 5 clases automáticamente
class_names = list(test_gen.class_indices.keys())

# --- MATRIZ DE CONFUSIÓN ---
# Cada fila representa la clase real, cada columna la clase predicha.
# Los valores en la diagonal son aciertos; fuera de ella, errores.
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names)
plt.title('Matriz de Confusión — EfficientNetB0')
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción del Modelo')
plt.xticks(rotation=45)
plt.tight_layout()
save_path = FIGURES_DIR / "matriz_confusion_EfficientNetB0.png"
plt.savefig(save_path, dpi=300)
plt.show()

# --- REPORTE DE CLASIFICACIÓN ---
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_true, y_pred, target_names=class_names))

### Matriz de confusión en texto

Imprime la misma matriz como arreglo de NumPy. Es útil para copiar los números exactos al reporte. El script `tests/comparar_modelos.py` lee esta salida directamente del notebook para armar la tabla comparativa entre modelos.

In [ ]:
print(cm)

### Conteo de errores

`np.where(y_pred != y_true)` devuelve los índices de las imágenes mal clasificadas. Con ellos calculamos cuántos errores hubo y volvemos a obtener la accuracy de forma manual (debe coincidir con la de `model.evaluate`).

Los índices quedan guardados en `errors` para inspeccionarlos visualmente en la celda siguiente.

In [ ]:
import numpy as np

# Buscamos dónde la predicción no coincide con la realidad
errors = np.where(y_pred != y_true)[0]

print(f"\nTotal de errores encontrados: {len(errors)} de {len(y_true)} imágenes reales.")
print(f"   Precisión Global (Accuracy): {1 - len(errors)/len(y_true):.2%}")

### Inspección visual de errores

Los números dicen *cuánto* se equivoca el modelo; ver las imágenes ayuda a entender *por qué*.

**¿Qué hace este código?**
1.  Toma las rutas de las imágenes (`test_gen.filepaths`) en el mismo orden que las predicciones.
2.  Muestra hasta **5 errores**, cada uno con su etiqueta real y la predicción del modelo (título en rojo).
3.  Si no hubiera errores, lo indica con un mensaje.

**¿Para qué sirve?** Suele revelar patrones: fotos con poca luz, objetos parcialmente fuera de cuadro, materiales parecidos (vidrio transparente vs. plástico transparente) o incluso imágenes mal etiquetadas en el dataset.

> ℹ️ Se muestran siempre los **primeros** 5 errores del set (en orden de carpeta), no una muestra aleatoria.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Extraemos las rutas de los archivos reales y los nombres de las clases
filepaths = test_gen.filepaths
class_names = list(test_gen.class_indices.keys())

# Cuántos errores queremos ver en pantalla (máximo 5 para no saturar)
num_errores_a_mostrar = min(5, len(errors))

if num_errores_a_mostrar > 0:
    print(f"\nMostrando {num_errores_a_mostrar} ejemplos de clasificaciones incorrectas:")
    plt.figure(figsize=(15, 5))
    
    for i in range(num_errores_a_mostrar):
        # Índice del error en el dataset
        idx = errors[i]
        
        # Ruta de la imagen real
        img_path = filepaths[idx]
        
        # Etiquetas correspondientes
        etiqueta_real = class_names[y_true[idx]]
        etiqueta_predicha = class_names[y_pred[idx]]
        
        # Cargar y mostrar la imagen
        img = mpimg.imread(img_path)
        plt.subplot(1, num_errores_a_mostrar, i + 1)
        plt.imshow(img)
        
        # El título mostrará en rojo la equivocación
        plt.title(f"Real: {etiqueta_real}\nPredicción: {etiqueta_predicha}", color='red')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("¡Excelente trabajo! El modelo no cometió ningún error en el set de pruebas.")

### Accuracy por clase

Para cada clase se filtran solo las imágenes que realmente pertenecen a ella (`mask = y_true == i`) y se calcula qué porcentaje acertó el modelo. Este valor es igual al **recall** de esa clase en el reporte de clasificación.

Permite identificar rápidamente qué categorías le cuestan más al modelo y es uno de los datos usados en el reporte comparativo entre modelos.

In [ ]:
# Accuracy por clase — permite identificar qué categorías le cuestan más al modelo.
# Esta información es útil para el reporte comparativo entre modelos.
print("\nAccuracy por clase:")
print("-" * 35)
for i, class_name in enumerate(class_names):
    mask = y_true == i
    class_acc = np.mean(y_pred[mask] == y_true[mask])
    print(f"  {class_name:<12}: {class_acc*100:.2f}%")